In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
import pickle

# 1. Nuestro corpus de texto (ejemplo pequeño para la demostración)
texto = """
El gato está sobre el techo.
El perro juega en el jardín.
El gato come pescado fresco.
El perro duerme todo el día.
"""

# 2. Inicializar y entrenar el Tokenizer
# El Tokenizer convertirá cada palabra en un número único.
tokenizer = Tokenizer()
# Ajustamos el tokenizer a nuestro texto (crea el diccionario)
tokenizer.fit_on_texts([texto])

# Obtenemos el tamaño del vocabulario (sumamos 1 por un requisito técnico de Keras para el padding)
vocab_size = len(tokenizer.word_index) + 1
print(f"Tamaño del vocabulario: {vocab_size} palabras únicas.")

# 3. Generar secuencias de N-gramas
# Esto crea las secuencias de aprendizaje. 
# Si la frase es "El gato está", genera: ["El gato"], ["El gato está"]
input_sequences = []
for line in texto.split('\n'):
    # Convierte la línea de texto a una lista de números
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# 4. Padding (Igualar tamaños)
# La red neuronal necesita que todas las entradas tengan la misma longitud.
# Rellenamos con ceros (0) a la izquierda las secuencias más cortas.
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# 5. Separar X (entrada) e Y (etiqueta a predecir)
# X son todas las palabras menos la última. Y es la última palabra.
X, y = input_sequences[:,:-1], input_sequences[:,-1]

# Convertimos 'y' al formato One-Hot Encoding (necesario para la capa Softmax)
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

# 6. Guardar el Tokenizer para usarlo después en la aplicación final
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("¡Procesamiento completado!")
print(f"Forma de X (Entradas): {X.shape}")
print(f"Forma de Y (Etiquetas): {y.shape}")